# CALM — Adaptive Multimodal Wildfire Monitoring

**Hệ thống AI giám sát cháy rừng thích ứng đa phương thức**, theo kiến trúc agentic (URSA):

- **Plan-driven execution:** PlanningAgent → RouterAgent → ExecutionAgent chạy từng step (không còn 2 pipeline cứng)
- **URSA 3 node:** Generator → Reflector → Formalizer
- **RSEN:** Reflexive Structured Experts Network (Weather + Geo analysts song song)
- **Evidence Evaluator:** chống hallucination; QA nhận `pre_retrieved` tránh duplicate
- **Safety Check** trước mỗi lệnh tool

**Các agents:** PlanningAgent | RouterAgent | DataKnowledgeAgent | RSENModule | WildfireQAAgent | ExecutionAgent | EvaluatorAgent

## 0. Setup — Môi trường và LLM

In [1]:
# Đảm bảo có thể import calm (khi mở notebook từ thư mục calm/ hoặc myProject/)
import sys
import warnings
warnings.filterwarnings("ignore", category=RuntimeWarning, module="duckduckgo_search")
warnings.filterwarnings("ignore", message="TqdmWarning|IProgress|BertModel|UNEXPECTED")

from pathlib import Path
_root = Path.cwd()
if _root.name == "calm":
    _src = _root / "src"
else:
    _src = _root / "calm" / "src"
if _src.exists() and str(_src) not in sys.path:
    sys.path.insert(0, str(_src))

# Nạp biến môi trường từ .env
from calm.utils.env_loader import load_env
load_env(_root / ".env")
if _root.name != "calm" and (_root / "calm").exists():
    load_env(_root / "calm" / ".env")
else:
    load_env()


In [2]:
import os
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model="arcee-ai/trinity-large-preview:free",
    max_completion_tokens=10000,
    openai_api_key=os.environ["OPENROUTER_API_KEY"],
    openai_api_base="https://openrouter.ai/api/v1",
)

/home/hungvu/code/khanh/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
response = llm.invoke("Explain the attention mechanism in transformers")
print(response.content)

The attention mechanism in transformers is a key innovation that allows models to weigh the importance of different words when processing language. Here's how it works:

When a transformer processes a word, it creates three vectors for each word in the sequence: a query vector (Q), a key vector (K), and a value vector (V). These vectors are derived from the word's embedding through learned linear transformations.

The attention calculation works like this: each query vector is compared to all key vectors using a dot product. This produces a score showing how relevant each word is to the current word being processed. These scores are then scaled down (typically by dividing by the square root of the key vector dimension) and passed through a softmax function to create weights that sum to 1.

These weights are then used to create a weighted sum of the value vectors. The result is a new representation where each word is now informed by all other words in the sequence, with more important w

## 1. Planning Agent (URSA – 3-node, Table A.1)

Theo mô tả trong paper, **Planning Agent** của URSA gồm 3 thành phần chính:

1. **Generator**  
   - Nhận `user query`.  
   - Sinh ra kế hoạch thực hiện ở dạng ngôn ngữ tự nhiên.

2. **Reflector**  
   - Đánh giá kế hoạch do Generator tạo ra.  
   - Kiểm tra các tiêu chí:
     - **Clarity** (tính rõ ràng)
     - **Completeness** (tính đầy đủ)
     - **Feasibility** (tính khả thi)
     - **Safety** (tính an toàn)
   - Nếu kế hoạch đạt yêu cầu → trả về nhãn **`[APPROVED]`**.  
   - Nếu chưa đạt → phản hồi để Generator chỉnh sửa.

3. **Formalizer**  
   - Chuyển kế hoạch đã được duyệt thành định dạng **JSON `plan_steps`**.  
   - Quá trình có thể lặp lại tối đa **`f_max` lần thử** nếu việc chuẩn hóa thất bại.

> Prompt cho các thành phần này được thiết kế theo **Appendix A – Table A.1** của paper.

In [4]:
from calm.agents.planning_agent import PlanningAgent

agent = PlanningAgent(llm=llm, config={}, n_max=3, f_max=3)
query = "Wildfire risk assessment for Amazon region next 7 days"
result = agent.invoke(query)

plan = result.get("final_output") or []
print("Approved:", result.get("approved"))
print("Số bước kế hoạch:", len(plan))
for i, step in enumerate(plan, 1):
    prompt = step.get("prompt", "")
    print(f"  Bước {i}: {step.get('step_id', '?')} | action={step.get('action')} | agent={step.get('agent')}")
    if prompt:
        print(f"       prompt: {prompt}")

Approved: True
Số bước kế hoạch: 3
  Bước 1: step-1 | action=retrieve_knowledge | agent=data_knowledge
  Bước 2: step-2 | action=web_search | agent=qa
  Bước 3: step-3 | action=prediction_reasoning | agent=prediction


### 1.1 RouterAgent — Task routing thay keyword fallback

RouterAgent xác định `task_type`, `confidence`, `required_artifacts` từ plan + query. Fallback keyword khi LLM fail.

In [ ]:
from calm.agents.router_agent import RouterAgent

router = RouterAgent(llm=llm, config={})
routing = router.route(query, plan)
print("Task type:", routing.task_type)
print("Confidence:", routing.confidence)
print("Required artifacts:", routing.required_artifacts)
print("Reasoning:", routing.reasoning)

## 2. DataKnowledgeAgent — Thu thập, Trích xuất, Truy xuất (§4.2)

**DataKnowledgeAgent** thực hiện pipeline: **Collection → Extraction → Retrieval**.

- **GeocodingTool:** Chỉ cần ghi địa chỉ (California, Amazon region...) → tool tự tìm tọa độ.
- **Collect:** GEE, Copernicus CDS, Web Search, ArXiv (có dedup check FR-D05).
- **Extract:** LLM trích xuất `factual_statements` và `causal_relationships` từ dữ liệu thô.
- **Retrieve:** Lưu tri thức vào ChromaDB, trả về dict (không phải DataFrame).

In [27]:
import os
import warnings
import importlib
from pathlib import Path

from dotenv import load_dotenv

warnings.filterwarnings("ignore", category=RuntimeWarning, module="duckduckgo_search")
load_dotenv(override=True)

True

In [28]:
from calm.agents.data_knowledge_agent import DataKnowledgeAgent
from calm.agents.memory_agent import MemoryAgent
from calm.memory.chroma_store import ChromaMemoryStore
from calm.tools.safety_check import SafetyChecker
from calm.tools.web_search import WebSearchTool
from calm.tools.earth_engine import EarthEngineTool
from calm.tools.arxiv_tool import ArXivTool

In [29]:
# Imports đã có ở cell trên; tiếp theo: cấu hình tools theo env

In [31]:
def env_flag(name: str, default: bool = False) -> bool:
    value = os.getenv(name, str(default)).strip().lower()
    return value in {"1", "true", "yes", "on"}


def env_int(name: str, default: int) -> int:
    try:
        return int(os.getenv(name, default))
    except Exception:
        return default


def load_optional_copernicus_class():
    candidates = [
        ("calm.tools.copernicus_tool", "CopernicusTool"),
        ("calm.tools.copernicus", "CopernicusTool"),
        ("calm.tools.copernicus_cds", "CopernicusTool"),
    ]

    for module_name, class_name in candidates:
        try:
            module = importlib.import_module(module_name)
            cls = getattr(module, class_name)
            return cls, module_name, class_name
        except Exception:
            continue

    return None, None, None


def write_cdsapirc_from_env():
    api_key = os.getenv("COPERNICUS_API_KEY", "").strip()
    api_url = os.getenv("COPERNICUS_URL", "https://cds.climate.copernicus.eu/api").strip()

    if not api_key:
        return False

    rc_path = Path.home() / ".cdsapirc"
    rc_path.write_text(
        f"url: {api_url}\n"
        f"key: {api_key}\n",
        encoding="utf-8",
    )
    return True

In [32]:
ENABLE_GEE = env_flag("ENABLE_GEE", True)
ENABLE_COPERNICUS = env_flag("ENABLE_COPERNICUS", True)
ENABLE_WEB_SEARCH = env_flag("ENABLE_WEB_SEARCH", True)
ENABLE_ARXIV = env_flag("ENABLE_ARXIV", True)

GEE_PROJECT_ID = os.getenv("GEE_PROJECT_ID", "").strip()
COPERNICUS_API_KEY = os.getenv("COPERNICUS_API_KEY", "").strip()

MAX_NEWS_RESULTS = env_int("MAX_NEWS_RESULTS", 3)
MAX_ARXIV_PAPERS = env_int("MAX_ARXIV_PAPERS", 3)
DEDUP_CHECK = env_flag("DEDUP_CHECK", False)

print("=== ENV CONFIG ===")
print("ENABLE_GEE =", ENABLE_GEE)
print("ENABLE_COPERNICUS =", ENABLE_COPERNICUS)
print("ENABLE_WEB_SEARCH =", ENABLE_WEB_SEARCH)
print("ENABLE_ARXIV =", ENABLE_ARXIV)
print("GEE_PROJECT_ID =", GEE_PROJECT_ID or "(empty)")
print("COPERNICUS_API_KEY =", "***set***" if COPERNICUS_API_KEY else "(empty)")
print("MAX_NEWS_RESULTS =", MAX_NEWS_RESULTS)
print("MAX_ARXIV_PAPERS =", MAX_ARXIV_PAPERS)
print("DEDUP_CHECK =", DEDUP_CHECK)

=== ENV CONFIG ===
ENABLE_GEE = True
ENABLE_COPERNICUS = True
ENABLE_WEB_SEARCH = True
ENABLE_ARXIV = True
GEE_PROJECT_ID = utilitarian-bee-490908-r7
COPERNICUS_API_KEY = ***set***
MAX_NEWS_RESULTS = 3
MAX_ARXIV_PAPERS = 3
DEDUP_CHECK = False


In [33]:
safety = SafetyChecker(llm=llm)
print("SafetyChecker đã tạo xong")

SafetyChecker đã tạo xong


In [34]:
web = None

if ENABLE_WEB_SEARCH:
    web = WebSearchTool(
        safety_checker=safety,
        config={"max_news_results": MAX_NEWS_RESULTS},
    )

print("web_search:", "ON" if web else "OFF")

web_search: ON


In [35]:
gee = None

if ENABLE_GEE:
    if GEE_PROJECT_ID:
        gee = EarthEngineTool(
            safety_checker=safety,
            config={"gee_project": GEE_PROJECT_ID},
        )
    else:
        print("Bỏ qua GEE: thiếu GEE_PROJECT_ID")

print("earth_engine:", "ON" if gee else "OFF")

earth_engine: ON


In [36]:
arxiv_tool = None

if ENABLE_ARXIV:
    arxiv_tool = ArXivTool(
        safety_checker=safety,
        config={"max_arxiv_papers": MAX_ARXIV_PAPERS},
    )

print("arxiv:", "ON" if arxiv_tool else "OFF")

arxiv: ON


In [37]:
copernicus = None

if ENABLE_COPERNICUS:
    CopernicusTool, module_name, class_name = load_optional_copernicus_class()

    if not COPERNICUS_API_KEY:
        print("Bỏ qua Copernicus: thiếu COPERNICUS_API_KEY")
    elif CopernicusTool is None:
        print("Bỏ qua Copernicus: không tìm thấy class CopernicusTool trong repo")
        print("Hãy kiểm tra lại import path của tool Copernicus")
    else:
        try:
            write_cdsapirc_from_env()
        except Exception as e:
            print("Không ghi được ~/.cdsapirc:", e)

        cop_config = {
            "api_key": COPERNICUS_API_KEY,
            "url": os.getenv("COPERNICUS_URL", "https://cds.climate.copernicus.eu/api"),
        }

        try:
            copernicus = CopernicusTool(
                safety_checker=safety,
                config=cop_config,
            )
            print(f"Copernicus loaded via {module_name}.{class_name}")
        except TypeError:
            try:
                copernicus = CopernicusTool(config=cop_config)
                print(f"Copernicus loaded via {module_name}.{class_name} (no safety_checker)")
            except Exception as e:
                print("Khởi tạo Copernicus thất bại:", e)
        except Exception as e:
            print("Khởi tạo Copernicus thất bại:", e)

print("copernicus:", "ON" if copernicus else "OFF")

Copernicus loaded via calm.tools.copernicus.CopernicusTool
copernicus: ON


In [38]:
print("=== TOOL STATUS ===")
print("web_search  :", "ON" if web else "OFF")
print("earth_engine:", "ON" if gee else "OFF")
print("copernicus  :", "ON" if copernicus else "OFF")
print("arxiv       :", "ON" if arxiv_tool else "OFF")

=== TOOL STATUS ===
web_search  : ON
earth_engine: ON
copernicus  : ON
arxiv       : ON


In [39]:
_store = ChromaMemoryStore(
    collection_name="calm_data_memory",
    persist_directory=".chroma",
    k=3,
)

memory = MemoryAgent(long_term_store=_store)

print("Memory đã sẵn sàng")

Memory đã sẵn sàng


In [40]:
from calm.tools.geocoding import GeocodingTool
geocoding = GeocodingTool(safety_checker=safety, config={})
tools = {
    "earth_engine": gee,
    "copernicus": copernicus,
    "web_search": web,
    "arxiv": arxiv_tool,
    "geocoding": geocoding,
}

print({k: ("ON" if v is not None else "OFF") for k, v in tools.items()})

{'earth_engine': 'ON', 'copernicus': 'ON', 'web_search': 'ON', 'arxiv': 'ON'}


In [41]:
data_agent = DataKnowledgeAgent(
    llm=llm,
    tools=tools,
    memory_store=memory,
    config={
        "dedup_check": DEDUP_CHECK,
        "max_news_results": MAX_NEWS_RESULTS,
        "max_arxiv_papers": MAX_ARXIV_PAPERS,
    },
)

print("DataKnowledgeAgent đã tạo xong")

DataKnowledgeAgent đã tạo xong


In [42]:
# Chỉ cần ghi địa chỉ — GeocodingTool tự tìm tọa độ
params = {
    "location": "California, USA",  # hoặc dict {"lat": 36.77, "lon": -119.41}
    "time_range": {"start": "2024-01-01", "end": "2024-12-31"},
}

query = "wildfire causes in California 2024"

print("Query:", query)
print("Params (địa chỉ văn bản → tool tự geocode):", params)

Query: wildfire causes in California 2024
Params: {'location': {'lat': 36.7783, 'lon': -119.4179}, 'time_range': {'start': '2024-01-01', 'end': '2024-12-31'}}


In [43]:
result = data_agent.retrieve(query, parameters=params)

print("retrieve xong")
print(type(result))

Safety check failed with error: Error code: 429 - {'error': {'message': 'Rate limit exceeded: free-models-per-day. Add 10 credits to unlock 1000 free model requests per day', 'code': 429, 'metadata': {'headers': {'X-RateLimit-Limit': '50', 'X-RateLimit-Remaining': '0', 'X-RateLimit-Reset': '1774137600000'}, 'provider_name': None}}, 'user_id': 'user_38eFibBaOsM8LFJwMbOM2dAYpKh'}. Treating as unsafe.
GEE collection failed: [UNSAFE] Safety check blocked: GEE fetch_satellite_stats location={'lat': 36.7783, 'lon': -119.4179} time_range={'start': '2024-01-01', 'end': '2024-12-31'} product=LANDSAT/LC08/C02/T1_L2
Safety check failed with error: Error code: 429 - {'error': {'message': 'Rate limit exceeded: free-models-per-day. Add 10 credits to unlock 1000 free model requests per day', 'code': 429, 'metadata': {'headers': {'X-RateLimit-Limit': '50', 'X-RateLimit-Remaining': '0', 'X-RateLimit-Reset': '1774137600000'}, 'provider_name': None}}, 'user_id': 'user_38eFibBaOsM8LFJwMbOM2dAYpKh'}. Treat

retrieve xong
<class 'dict'>


In [44]:
print("=" * 50)
print("RETRIEVAL SUMMARY")
print("=" * 50)

summary = result.get("retrieval_summary", {})
print("Query:", summary.get("original_query", "N/A"))

if result.get("dedup"):
    print("Source: cache (dedup hit)")
else:
    sources = [r.get("source", "?") for r in result.get("retrieved_data", [])]
    print(f"Nguồn ({len(sources)}):", ", ".join(sources) if sources else "không có")

RETRIEVAL SUMMARY
Query: wildfire causes in California 2024
Nguồn (0): không có


In [45]:
print("\n" + "-" * 50)
print("RETRIEVED DATA")
print("-" * 50)

for i, item in enumerate(result.get("retrieved_data", []), 1):
    src = item.get("source", "?")
    dc = item.get("data_content", {})

    if isinstance(dc, dict):
        title = (dc.get("title") or dc.get("name") or dc.get("summary", ""))[:60]
    else:
        title = str(dc)[:60]

    print(f"  {i}. [{src}] {title}...")


--------------------------------------------------
RETRIEVED DATA
--------------------------------------------------


In [46]:
print("\n" + "-" * 50)
print("EXTRACTED KNOWLEDGE")
print("-" * 50)

ek = result.get("extracted_knowledge", {})
facts = ek.get("factual_statements", [])
causal = ek.get("causal_relationships", [])

print(f"\nFactual statements ({len(facts)}):")
for j, s in enumerate(facts[:5], 1):
    txt = str(s)
    print(f"  {j}. {txt[:100]}..." if len(txt) > 100 else f"  {j}. {txt}")

if len(facts) > 5:
    print(f"  ... và {len(facts) - 5} câu khác")

print(f"\nCausal relationships ({len(causal)}):")
for j, c in enumerate(causal[:5], 1):
    txt = str(c)
    print(f"  {j}. {txt[:100]}..." if len(txt) > 100 else f"  {j}. {txt}")

if len(causal) > 5:
    print(f"  ... và {len(causal) - 5} quan hệ khác")


--------------------------------------------------
EXTRACTED KNOWLEDGE
--------------------------------------------------

Factual statements (0):

Causal relationships (0):


In [47]:
print("\n" + "=" * 50)
print("TỔNG KẾT:")
print(f"  retrieved_data: {len(result.get('retrieved_data', []))} mục")
print(f"  factual_statements: {len(facts)}")
print(f"  causal_relationships: {len(causal)}")
print("=" * 50)


TỔNG KẾT:
  retrieved_data: 0 mục
  factual_statements: 0
  causal_relationships: 0


## 3. RSEN — Reflexive Structured Experts Network  
*(Equations 17–22, Appendix A Tables A.6–A.8)*

RSEN được thiết kế theo mô hình **đa chuyên gia phản tư (reflexive multi-expert system)**, trong đó các chuyên gia hoạt động **song song và độc lập** để phân tích các khía cạnh khác nhau của truy vấn.

### Kiến trúc thành phần

Hệ thống bao gồm ba tác nhân chính:

- **Weather Analyst**  
  Chuyên gia phân tích **khí tượng**, đánh giá các yếu tố liên quan đến thời tiết như:
  - nhiệt độ
  - lượng mưa
  - điều kiện khí hậu
  - các hiện tượng thời tiết đặc biệt

- **Geo Analyst**  
  Chuyên gia phân tích **địa lý**, xem xét các yếu tố:
  - đặc điểm địa hình
  - vị trí địa lý
  - điều kiện môi trường khu vực
  - tính hợp lý về mặt không gian

- **Ops Coordinator**  
  Tác nhân điều phối trung tâm có nhiệm vụ:
  - tổng hợp báo cáo từ các chuyên gia
  - đánh giá tính nhất quán giữa các phân tích
  - đưa ra quyết định cuối cùng:
    - **Plausible** — thông tin hợp lý  
    - **Implausible** — thông tin không hợp lý

### Cơ chế phản tư (Reflexion)

Hệ thống sử dụng **bộ nhớ dài hạn** dựa trên **ChromaDB**:

- **Collection:** `calm_rsen_memory`
- **Chức năng:** lưu trữ các trường hợp phân tích trước đó
- **Truy xuất:** sử dụng **top-k retrieval** để tìm các kinh nghiệm liên quan

Cơ chế này cho phép các chuyên gia:
- tham chiếu các phân tích trước đây
- cải thiện chất lượng lập luận
- tăng tính nhất quán và độ tin cậy của quyết định cuối cùng.

In [48]:
import warnings
warnings.filterwarnings("ignore", message="TqdmWarning|IProgress|BertModel|UNEXPECTED")

from calm.agents.memory_agent import MemoryAgent
from calm.agents.rsen_module import RSENModule
from calm.memory.chroma_store import ChromaMemoryStore

_store = ChromaMemoryStore(collection_name="calm_rsen_memory", persist_directory=".chroma", k=3)
memory = MemoryAgent(long_term_store=_store)
rsen = RSENModule(llm=llm, memory_store=memory, k=3)

result = rsen.validate(
    prediction={"risk_level": "High", "confidence": 0.8},
    met_data={"temperature": 35.0, "humidity": 0.2, "wind_speed": 15},
    spatial_data={"fuel_type": "Shrubland", "slope": 25, "elevation": 500},
)

print("Validation:", result.get("validation_decision"))
print("Rationale:", (result.get("final_rationale") or "")[:400])

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 9192.16it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


RateLimitError: Error code: 429 - {'error': {'message': 'Rate limit exceeded: free-models-per-day. Add 10 credits to unlock 1000 free model requests per day', 'code': 429, 'metadata': {'headers': {'X-RateLimit-Limit': '50', 'X-RateLimit-Remaining': '0', 'X-RateLimit-Reset': '1774137600000'}, 'provider_name': None}}, 'user_id': 'user_38eFibBaOsM8LFJwMbOM2dAYpKh'}

## 4. Wildfire QA Agent — Evidence Evaluator  
*(Appendix A Tables A.2–A.5)*

Wildfire QA Agent là tác nhân hỏi–đáp chuyên biệt cho miền **cháy rừng**, được thiết kế theo nguyên tắc **evidence-driven reasoning** nhằm giảm thiểu **hallucination** của mô hình ngôn ngữ.

Hệ thống chỉ tạo câu trả lời khi có **bằng chứng hỗ trợ đầy đủ**, nếu không sẽ tiếp tục tìm kiếm dữ liệu bổ sung.

---

### 4.1 Kiến trúc Pipeline

Pipeline xử lý gồm các bước tuần tự sau:

1. **Retrieve — DataKnowledgeAgent**

   Truy vấn ban đầu được gửi tới **DataKnowledgeAgent** để thực hiện truy xuất tri thức từ các nguồn:

   - vector database (ChromaDB)
   - tài liệu tri thức nội bộ
   - bộ nhớ hệ thống (memory store)

   Kết quả là tập hợp **evidence candidates** liên quan đến câu hỏi.

---

2. **Evidence Evaluation**

   Tác nhân **Evidence Evaluator** đánh giá chất lượng và mức độ liên quan của các bằng chứng.

   Các tiêu chí đánh giá bao gồm:

   - độ liên quan ngữ nghĩa (semantic relevance)
   - độ tin cậy của nguồn
   - tính đầy đủ của thông tin
   - sự nhất quán giữa các bằng chứng

   Điểm đánh giá được so sánh với tham số:


In [6]:
import warnings
warnings.filterwarnings("ignore", category=RuntimeWarning, module="duckduckgo_search")

from calm.agents.data_knowledge_agent import DataKnowledgeAgent
from calm.agents.memory_agent import MemoryAgent
from calm.agents.qa_agent import WildfireQAAgent
from calm.memory.chroma_store import ChromaMemoryStore
from calm.tools.safety_check import SafetyChecker
from calm.tools.web_search import WebSearchTool

safety = SafetyChecker(llm=llm)
web = WebSearchTool(safety_checker=safety, config={"max_news_results": 5})
_store = ChromaMemoryStore(collection_name="calm_qa_memory", persist_directory=".chroma", k=3)
memory = MemoryAgent(long_term_store=_store)
tools = {"earth_engine": None, "copernicus": None, "web_search": web, "arxiv": None}
data_agent = DataKnowledgeAgent(
    llm=llm, tools=tools, memory_store=memory, config={"dedup_check": True},
)
qa = WildfireQAAgent(
    llm=llm,
    data_agent=data_agent,
    web_search_tool=web,
    memory_store=memory,
    config={"evidence_threshold": 0.65},
)

result = qa.invoke("What caused the 2023 Canadian wildfires?")
out = result.get("final_output") or {}
print("Answer:", out.get("answer", out))
print("Citations:", out.get("citations", []))
print("Confidence:", out.get("confidence"))

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8026.11it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
/home/hungvu/code/khanh/v2/calm-/src/calm/tools/web_search.py:41: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  results = list(DDGS().text(query, max_results=max_results))
/home/hungvu/code/khanh/v2/calm-/src/calm/tools/web_search.py:41: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  results = list(DDGS().text(query, max_results=max_results))
/home/hungvu/code/khanh/v2/calm-/src/calm/tools/web_search.py:41: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to

Answer: The 2023 Canadian wildfires were primarily driven by a combination of extreme climatic conditions and human factors. Key contributors include:

1. **Climatic Factors**: 
   - **Drought and Dry Conditions**: Many regions in Canada experienced unusually dry conditions leading up to the wildfire season. This was exacerbated by prolonged periods of below-average precipitation.
   - **Heatwaves**: The summer of 2023 saw record-breaking temperatures, which created an environment conducive to wildfire ignition and spread. Heatwaves significantly increased the flammability of vegetation.
   - **Lightning Strikes**: A significant proportion of wildfires ignited due to lightning strikes, which were frequent during thunderstorms in the region during the summer months.

2. **Human Activities**: 
   - **Arson and Accidental Ignitions**: There were reports of wildfires that were either set intentionally or ignited through human activities such as campfires or discarded cigarettes.
   - **Lan

## 5. ExecutionAgent — Thực thi từng bước kế hoạch

**ExecutionAgent** nhận `step` từ plan và route sang đúng tool:
- `data_knowledge` → DataKnowledgeAgent.retrieve()
- `qa` → WildfireQAAgent.invoke(pre_retrieved) khi có data_result
- `prediction` → PredictionReasoningAgent.predict()
- `rsen` → RSENModule.validate()
- `web_search` → WebSearchTool.search()

Mỗi lệnh tool đều qua **SafetyChecker** trước khi thực thi.

In [ ]:
import warnings
warnings.filterwarnings("ignore", message="TqdmWarning|IProgress|BertModel|UNEXPECTED")
warnings.filterwarnings("ignore", category=RuntimeWarning, module="duckduckgo_search")

from calm.agents.execution_agent import ExecutionAgent
from calm.agents.data_knowledge_agent import DataKnowledgeAgent
from calm.agents.prediction_reasoning_agent import PredictionReasoningAgent
from calm.agents.qa_agent import WildfireQAAgent
from calm.agents.rsen_module import RSENModule
from calm.memory.chroma_store import ChromaMemoryStore
from calm.tools.safety_check import SafetyChecker
from calm.tools.web_search import WebSearchTool

# Khởi tạo tools cho ExecutionAgent
safety = SafetyChecker(llm=llm)
web = WebSearchTool(safety_checker=safety, config={"max_news_results": 3})
_store_qa = ChromaMemoryStore(collection_name="calm_qa_memory", persist_directory=".chroma", k=3)
_store_rsen = ChromaMemoryStore(collection_name="calm_rsen_memory", persist_directory=".chroma", k=3)
mem_qa = MemoryAgent(long_term_store=_store_qa)
mem_rsen = MemoryAgent(long_term_store=_store_rsen)
tools_dict = {
    "earth_engine": None, "copernicus": None, "web_search": web, "arxiv": None,
}
data_agent = DataKnowledgeAgent(llm=llm, tools=tools_dict, memory_store=mem_qa, config={"dedup_check": True})
qa_agent = WildfireQAAgent(llm=llm, data_agent=data_agent, web_search_tool=web, memory_store=mem_qa, config={"evidence_threshold": 0.65})
rsen_agent = RSENModule(llm=llm, memory_store=mem_rsen, k=3)
prediction_agent = PredictionReasoningAgent(model_runner=None, config={})

exec_tools = {
    "data_knowledge": data_agent,
    "qa": qa_agent,
    "rsen": rsen_agent,
    "prediction": prediction_agent,
    "web_search": web,
}
executor = ExecutionAgent(llm=llm, tools=exec_tools, safety_checker=safety)

# Test 1: Step RSEN (gọi LLM qua RSEN)
step_rsen = {"step_id": "step-1", "action": "validate_prediction", "agent": "rsen", "parameters": {}}
ctx = {"query": "risk?", "prediction": {"risk_level": "High", "confidence": 0.8}, "met_data": {"temperature": 32, "humidity": 0.25}, "spatial_data": {"fuel_type": "Grassland", "slope": 15}}
res_rsen = executor.execute_step(step_rsen, ctx)
print("RSEN step:", res_rsen.get("validation_decision", res_rsen.get("error", "ok")))

# Test 2: Step web_search (gọi tool)
step_web = {"step_id": "step-2", "action": "web_search", "agent": "web_search", "parameters": {"query": "California wildfire 2024"}}
res_web = executor.execute_step(step_web, {"query": "California wildfire 2024"})
print("Web step results:", len(res_web.get("results", [])))

### 5.1 CALMOrchestrator — Plan-driven Full Pipeline

Điểm vào duy nhất: `orchestrator.run(query)`. PlanningAgent → RouterAgent → ExecutionAgent chạy từng step.

In [ ]:
from calm.orchestrator import CALMOrchestrator
from calm.memory.chroma_store import ChromaMemoryStore
from calm.tools.safety_check import SafetyChecker
from calm.tools.web_search import WebSearchTool

_store = ChromaMemoryStore(collection_name="calm_orch_memory", persist_directory=".chroma", k=3)
_web = WebSearchTool(safety_checker=SafetyChecker(llm=llm), config={"max_news_results": 3})
tools = {"earth_engine": None, "copernicus": None, "web_search": _web, "arxiv": None}
orch = CALMOrchestrator.from_llm(llm=llm, memory_store=_store, tools=tools, config={})

result = orch.run("What causes wildfires in California?")
print("Task type:", result.get("task_type"))
print("Answer:", (result.get("answer", "") or str(result))[:300], "...")

## 6. Full Pipeline — Plan + LLM-as-a-Judge (Evaluator, §5.2)

Hệ thống sử dụng pipeline gồm hai thành phần chính: **Planning** và **Evaluation** nhằm đảm bảo chất lượng đầu ra của mô hình.

### 6.1 Planning

Từ **query** của người dùng, hệ thống tạo một **kế hoạch xử lý (plan)** mô tả các bước cần thực hiện, ví dụ:

- truy xuất dữ liệu (retrieval)
- phân tích thông tin
- gọi các công cụ cần thiết
- tổng hợp kết quả

Kế hoạch này giúp hệ thống xử lý truy vấn một cách có cấu trúc và logic.

### 6.2 Execution

Hệ thống thực thi kế hoạch đã tạo để sinh ra **candidate output** (kết quả hoặc câu trả lời ban đầu).

### 6.3 Evaluation — LLM-as-a-Judge

Kết quả đầu ra được chuyển đến **Evaluator Agent** để đánh giá chất lượng.  
Evaluator đóng vai trò **LLM-as-a-Judge**, chấm điểm câu trả lời theo **5 tiêu chí**, mỗi tiêu chí trong khoảng **0–100**:

| Tiêu chí | Mô tả |
|---|---|
| **Data Accuracy** | Độ chính xác và tính đúng đắn của dữ liệu |
| **Explainability** | Mức độ rõ ràng và khả năng giải thích của câu trả lời |
| **Jargon Avoidance** | Hạn chế sử dụng thuật ngữ chuyên môn khó hiểu |
| **Redundancy Avoidance** | Tránh lặp lại thông tin không cần thiết |
| **Citation Quality** | Chất lượng và độ tin cậy của các nguồn trích dẫn |

### 6.4 Passing Criterion

Điểm trung bình của năm tiêu chí được tính như sau:


average_score = mean(all_scores)


Một câu trả lời được chấp nhận khi:


average_score >= passing_score


Trong đó:


passing_score = 75

In [7]:
from calm.agents.evaluator_agent import EvaluatorAgent
from calm.agents.planning_agent import PlanningAgent

planner = PlanningAgent(llm=llm, config={})
evaluator = EvaluatorAgent(llm=llm, config={"passing_score": 75.0})

query = "Assess wildfire risk for California Central Valley next 14 days"
plan_result = planner.invoke(query)
plan = plan_result.get("final_output") or []

eval_result = evaluator.evaluate(
    response={"plan": plan, "query": query},
    query=query,
)

print("Plan steps:", len(plan))
print("Evaluation passed:", eval_result.get("passed"))
print("Average score:", eval_result.get("average_score"))
print("Scores:", eval_result.get("scores", {}))
if eval_result.get("recommendations"):
    print("Recommendations:", eval_result["recommendations"])

Plan steps: 4
Evaluation passed: False
Average score: 63.0
Scores: {'data_accuracy': 70, 'explainability': 75, 'jargon_avoidance': 80, 'redundancy_avoidance': 90, 'citation_quality': 50}
Recommendations: ['Include specific data and sources to improve data accuracy and citation quality.', 'Provide a summary of findings or insights from the web search and knowledge retrieval steps to enhance explainability.', 'Consider using simpler language for any technical jargon to make it more accessible to non-experts.']
